# Step 5 — Self-Supervised Pre-Training: `1D-ViT (SpectralMAE)` on High-Wavenumber (`HW: 2765–3200 cm⁻¹`)

This HPC JupyterLab notebook runs the **exact winning `SpectralMAE` architecture (`4,485,604` parameters)** from Deliverable 3.2 Table 2 (`patch_size = 100`, `10` patches, `50%` mask ratio, `8` encoder blocks `dim=256`, `1` decoder block `dim=128`, `500` epochs, `AdamW lr=1e-4`, Cosine Annealing) on the harmonized **`448,358`-spectrum High-Wavenumber corpus (`pretrain_vit_hw_1000.npz`)**.

> **Before running on HPC JupyterLab**:
> 1. Run `pip install -e .` once in the HPC terminal inside the repository root.
> 2. Upload `data/02_ready/pretrain/HW/pretrain_vit_hw_1000.npz` (`1.71 GB`) into `data/02_ready/pretrain/HW/`.

## 1. HPC Environment & GPU Verification
Verify CUDA GPU availability, VRAM, and mixed-precision (`bf16`/`fp16`) support.

In [ ]:
from pathlib import Path
import torch, yaml, numpy as np, pandas as pd
from IPython.display import Image, display

ROOT = Path('..').resolve()
print('PyTorch version :', torch.__version__)
print('CUDA available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device      :', torch.cuda.get_device_name(0))
    print('VRAM (GB)       :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2))
    print('BF16 supported  :', torch.cuda.is_bf16_supported())

## 2. Load Pre-Training Configuration (`configs/train/smae_hw_1000.yaml`)
Inspect the Deliverable 3.2 hyperparameters (`500` epochs, `batch_size=256`, `lr=1e-4`, `patch_size=100`, `mask_ratio=0.50`, `sampling_alpha=0.5`).

In [ ]:
CONFIG_PATH = ROOT / 'configs' / 'train' / 'smae_hw_1000.yaml'
cfg = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
print(yaml.safe_dump(cfg, sort_keys=False))

## 3. Inspect the Harmonized `HW` Pre-Training Corpus (`448,358` Spectra, `2765–3200 cm⁻¹`)
Verify exact spectrum counts, shared wavenumber grid, and `[0.0000, 1.0000]` scaling before launching training.

In [ ]:
npz_path = ROOT / cfg['data']['npz_path']
arc = np.load(npz_path, allow_pickle=True)
wn = arc['wavenumbers']
print(f"Loaded: {npz_path.name} | Shape: {arc['spectra'].shape} | Grid: {wn[0]:.2f}-{wn[-1]:.2f} cm^-1 ({wn.size} pts)")
for name, cnt in zip(arc['source_names'], arc['source_counts']):
    print(f'  - {str(name):22s}: {int(cnt):7,d} spectra')

## 4. Verify `SpectralMAE` Architecture & Parameter Count (`4,485,604` Parameters)
Instantiate `SpectralMAE` and run a single dummy forward pass to confirm shapes and exact Deliverable 3.2 parameter count.

In [ ]:
from spectrabreast.models import SpectralMAE

m_cfg = cfg['model']
model_check = SpectralMAE(
    spectrum_size=m_cfg['spectrum_size'], patch_size=m_cfg['patch_size'],
    embed_dim=m_cfg['embed_dim'], depth=m_cfg['depth'], num_heads=m_cfg['num_heads'],
    mlp_dim=m_cfg['mlp_dim'], decoder_embed_dim=m_cfg['decoder_embed_dim'],
    decoder_depth=m_cfg['decoder_depth'], decoder_num_heads=m_cfg['decoder_num_heads'],
    decoder_mlp_dim=m_cfg['decoder_mlp_dim'],
)
out = model_check(torch.rand(4, 1000), mask_ratio=m_cfg['mask_ratio'])
print(f"Total Parameters: {sum(p.numel() for p in model_check.parameters()):,d} | Loss: {out['loss'].item():.5f} | CosSim: {out['masked_cosine_sim'].item():.4f}")

## 5. Launch `1D-ViT (SMAE)` Pre-Training (`500` Epochs with Auto-Resume)
Calls `train_smae_from_config`. Automatically resumes from `last_checkpoint.pth` if interrupted, logs `history.csv`, saves `best_smae_model.pth`, and renders reconstruction progress snapshots at Epochs `1, 10, 50, 100, 250, 500`.

In [ ]:
from spectrabreast.training import train_smae_from_config

results = train_smae_from_config(
    config_path=CONFIG_PATH,
    repo_root=ROOT,
    resume=True,
    max_epochs_override=None,  # Set e.g. 5 for a quick smoke test!
)
print(f"Finished! Best Val Masked MSE = {results['best_val_mse']:.6f}")
print('Best checkpoint:', results['best_checkpoint'])

## 6. Visualize Learning Curves (Masked Reconstruction MSE & Cosine Similarity)
Exact Deliverable 3.2 Fig. 4 / Report_Pre_Training Fig. 3 layout.

In [ ]:
fig_dir = results['output_dir'] / 'figures'
display(Image(filename=str(fig_dir / 'smae_learning_curves.png')))

## 7. Compare Multi-Source Spectral Reconstructions: Epoch 1 vs Final Epoch
Exact Deliverable 3.2 Fig. 5 layout (Blue: Original Spectrum, Red Dashed: MAE Reconstruction, Yellow Shading: Masked Patches across all 5 HW sources).

In [ ]:
recon_figs = sorted(fig_dir.glob('smae_reconstruction_epoch_*.png'))
print('Saved reconstruction snapshots:', [p.name for p in recon_figs])
if recon_figs:
    display(Image(filename=str(recon_figs[0])))
    if len(recon_figs) > 1:
        display(Image(filename=str(recon_figs[-1])))

## 8. Post-Training Interactive Evaluation on Fresh Random `HW` Spectra
Matches `MAE_100_patches.ipynb` Cell 7: re-run this cell anytime to sample brand-new random spectra from all 5 `HW` cohorts and inspect live masked reconstructions.

In [ ]:
from spectrabreast.training import plot_smae_reconstructions
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_eval = model_check.to(device)
model_eval.load_state_dict(torch.load(results['best_checkpoint'], map_location=device, weights_only=True))
model_eval.eval()
origs, recons, masks = {}, {}, {}
with torch.no_grad():
    for sid, sname in enumerate(arc['source_names']):
        idx = int(np.random.choice(np.where(arc['source_id'] == sid)[0]))
        sp = arc['spectra'][idx].astype(np.float32)
        out_s = model_eval(torch.from_numpy(sp).unsqueeze(0).to(device), mask_ratio=m_cfg['mask_ratio'])
        origs[str(sname)], recons[str(sname)], masks[str(sname)] = sp, out_s['pred_spectra'][0].cpu().numpy(), out_s['mask'][0].cpu().numpy()
plot_smae_reconstructions(wn, origs, recons, masks, epoch=cfg['training']['epochs'], patch_size=m_cfg['patch_size'])